<a href="https://colab.research.google.com/github/Chetalam/cnn-skin-cancer-classification/blob/main/01_database_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
HAM_ROOT = PROJECT_ROOT / 'Datasets' / 'HAM10000'
PAD_ROOT = PROJECT_ROOT / 'Datasets' / 'PAD-UFES-20'
DB_ROOT = PROJECT_ROOT / 'Database'
MODEL_ROOT = PROJECT_ROOT / 'Models'

for folder in (HAM_ROOT, PAD_ROOT, DB_ROOT, MODEL_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

# Compatibility with your existing cells that concatenate path strings.
PROJECT_PATH = str(PROJECT_ROOT) + '/'
HAM_PATH = str(HAM_ROOT) + '/'
PAD_PATH = str(PAD_ROOT) + '/'
DATABASE_PATH = str(DB_ROOT / 'skin_cancer.db')
MODELS_PATH = str(MODEL_ROOT) + '/'

for name, value in {
    'Project': PROJECT_PATH,
    'HAM10000': HAM_PATH,
    'PAD-UFES-20': PAD_PATH,
    'Database file (to be created later)': DATABASE_PATH,
    'Models': MODELS_PATH,
}.items():
    print(f'{name}: {value}')

Project: /content/drive/MyDrive/cnn-skin-cancer-classification/
HAM10000: /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/
PAD-UFES-20: /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/PAD-UFES-20/
Database file (to be created later): /content/drive/MyDrive/cnn-skin-cancer-classification/Database/skin_cancer.db
Models: /content/drive/MyDrive/cnn-skin-cancer-classification/Models/


In [4]:
probe = PROJECT_ROOT / '_drive_write_check.txt'
if probe.exists():
    raise FileExistsError('A previous check file exists; inspect it before rerunning.')
probe.write_text('Drive write check successful', encoding='utf-8')
assert probe.read_text(encoding='utf-8') == 'Drive write check successful'
print('Drive read/write check passed:', probe)

Drive read/write check passed: /content/drive/MyDrive/cnn-skin-cancer-classification/_drive_write_check.txt


In [5]:
probe.unlink()
print('Temporary check file removed.')

Temporary check file removed.


In [6]:
import pandas as pd
from IPython.display import display

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
dataset_inventory = {}

for dataset_name, root in {'HAM10000': HAM_ROOT, 'PAD-UFES-20': PAD_ROOT}.items():
    files = [p for p in root.rglob('*') if p.is_file()]
    images = sorted(p for p in files if p.suffix.lower() in IMAGE_EXTENSIONS)
    csvs = sorted(p for p in files if p.suffix.lower() == '.csv')
    archives = [p for p in files if p.suffix.lower() in {'.zip', '.gz', '.tar'}]
    dataset_inventory[dataset_name] = {'images': images, 'csvs': csvs}
    print(f'\n{dataset_name}: {len(images)} image files; {len(csvs)} CSV files; '
          f'{len(archives)} archive files')
    print('CSV candidates:')
    for index, path in enumerate(csvs):
        print(index, path)
    print('First 3 images:')
    for path in images[:3]:
        print(path)
    if not images or not csvs:
        print('CHECK REQUIRED: extracted images and metadata CSV are both needed.')


HAM10000: 10033 image files; 5 CSV files; 0 archive files
CSV candidates:
0 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_metadata.csv
1 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/hmnist_28_28_L.csv
2 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/hmnist_28_28_RGB.csv
3 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/hmnist_8_8_L.csv
4 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/hmnist_8_8_RGB.csv
First 3 images:
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0024306.jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0024307.jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0024308.jpg

PAD-UFES-20: 2300 image files; 1 CSV files; 0 archive files
CSV candidates:
0 /content/drive/MyDrive/cnn

In [7]:
ham_metadata_path = Path('/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_metadata.csv')
pad_metadata_path = Path('/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/PAD-UFES-20/metadata.csv')

for name, path in {'HAM10000': ham_metadata_path,
                   'PAD-UFES-20': pad_metadata_path}.items():
    if not path.is_file():
        raise FileNotFoundError(f'{name}: select an existing metadata CSV: {path}')
    frame = pd.read_csv(path)
    print(f'\n{name}: {frame.shape[0]} rows, {frame.shape[1]} columns')
    print('Columns:', frame.columns.tolist())
    display(frame.head())
    print('Data types:')
    print(frame.dtypes)


HAM10000: 10015 rows, 7 columns
Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


Data types:
lesion_id        object
image_id         object
dx               object
dx_type          object
age             float64
sex              object
localization     object
dtype: object

PAD-UFES-20: 2298 rows, 26 columns
Columns: ['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father', 'background_mother', 'age', 'pesticide', 'gender', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'fitspatrick', 'region', 'diameter_1', 'diameter_2', 'diagnostic', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation', 'img_id', 'biopsed']


,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NaN,NEV,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1516_1765_530.png,False
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,5.0,BCC,TRUE,TRUE,FALSE,TRUE,TRUE,TRUE,PAT_46_881_939.png,True
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1545_1867_547.png,False
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1989_4061_934.png,False
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,5.0,BCC,TRUE,TRUE,FALSE,FALSE,TRUE,TRUE,PAT_684_1302_588.png,True


Data types:
patient_id              object
lesion_id                int64
smoke                   object
drink                   object
background_father       object
background_mother       object
age                      int64
pesticide               object
gender                  object
skin_cancer_history     object
cancer_history          object
has_piped_water         object
has_sewage_system       object
fitspatrick            float64
region                  object
diameter_1             float64
diameter_2             float64
diagnostic              object
itch                    object
grew                    object
hurt                    object
changed                 object
bleed                   object
elevation               object
img_id                  object
biopsed                   bool
dtype: object
